# Notebook 22c — Prompt Sensitivity
**AI Models and Agents for Business · Week 11 · Session 22**

A manager runs a test: "the AI classifies our customer reviews with 83% accuracy." The team deploys it. Later someone rewrites the prompt to be a bit clearer, and accuracy drops. **Nothing about the task changed.** Today you measure how much a model's answers move when the *wording* changes but the *meaning* does not.

### What you will do
1. Classify 12 customer reviews with **eight different prompts** that all ask for the same thing.
2. Compare accuracy across prompts, and find the **unstable** reviews.
3. Separate **prompt sensitivity** (changing the words) from **run-to-run randomness** (same words, again).
4. Test whether **examples in the prompt** make the model steadier.
5. Write one prompt variant of your own.

### Ground rules
- Temperature is set to **0** for the main experiment, so differences come from the *prompt*, not chance.
- Running all prompts takes about 2 to 4 minutes on the free tier. If you see a "rate limit" message, the notebook waits and continues.

### One-time setup: store your key in Colab Secrets
1. Click the **🔑 key icon** in the left sidebar of Colab.
2. Click **Add new secret**. Name: `GROQ_API_KEY`. Value: your key (it starts with `gsk_`).
3. Turn on **Notebook access** for this notebook.

Colab will remember the key for all your notebooks, so you never paste it into a cell. If you skip this, the setup cell will simply ask you to paste the key each time (the box hides what you type).

In [ ]:
# Run this cell first. Do not modify.
!pip install groq --quiet

import time, json, textwrap
from groq import Groq
import groq

def _get_key():
    """Look for the key in Colab Secrets first; otherwise ask for it (hidden as you type)."""
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        import getpass
        return getpass.getpass("Paste your Groq API key (starts with gsk_): ")

API_KEY = _get_key()
client = Groq(api_key=API_KEY)
MODEL = "openai/gpt-oss-20b"         # if Groq retires this name, change it here only

# Groq retired llama-3.1-8b-instant for free accounts in August 2026, so this notebook uses
# openai/gpt-oss-20b. It is a reasoning model: before it answers, it writes hidden "thinking"
# tokens. They count toward max_tokens and are billed as output tokens. The lines below keep
# that thinking short and add room for it, so every max_tokens in this notebook still means
# "room for the answer".
REASONING_EFFORT = "low"      # "low", "medium" or "high"
REASONING_HEADROOM = 300      # extra tokens reserved for the hidden thinking
_groq_create = client.chat.completions.create
def _create_with_reasoning(*args, **kwargs):
    if "gpt-oss" in str(kwargs.get("model", "")):
        kwargs.setdefault("reasoning_effort", REASONING_EFFORT)
        kwargs.setdefault("include_reasoning", False)
        if kwargs.get("max_tokens"):
            kwargs["max_tokens"] += REASONING_HEADROOM
    return _groq_create(*args, **kwargs)
client.chat.completions.create = _create_with_reasoning

def _retry_after(err, default=15):
    """Seconds the free tier asks us to wait (from the error's retry-after header), capped at 60."""
    try:
        return min(60.0, float(err.response.headers.get("retry-after", default)))
    except Exception:
        return float(default)

def ask(prompt, system=None, temperature=0.0, max_tokens=300, messages=None, model=None):
    """Send one request to the model and return the full response object.
    Waits and retries if the free tier says 'slow down'."""
    if messages is None:
        messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
    for attempt in range(6):
        try:
            return client.chat.completions.create(model=model or MODEL, messages=messages,
                                                  temperature=temperature, max_tokens=max_tokens)
        except groq.RateLimitError as err:
            wait = _retry_after(err)
            print(f"(rate limit hit - waiting {wait:.0f} seconds, then trying again)")
            time.sleep(wait)
    raise RuntimeError("Still rate limited. Wait a minute and re-run this cell.")

def say(prompt, **kw):
    """Same as ask(), but returns just the text of the answer."""
    return (ask(prompt, **kw).choices[0].message.content or "").strip()

print("✅ Setup complete. Model:", MODEL)

## Part A — The reviews

Twelve reviews with a "gold" label chosen by a person. **Two are genuinely mixed** (marked below), so the gold label is a judgment call. That is realistic: real labels are rarely perfect.

In [ ]:
REVIEWS = [
    ("Absolutely love this jacket, warm and light.", "positive"),
    ("Arrived broken. Complete waste of money.", "negative"),
    ("It's okay. Does what it says, nothing special.", "neutral"),
    ("Fast shipping and the fit is perfect.", "positive"),
    ("Zipper failed after two uses. Very disappointed.", "negative"),
    ("Product matches the description.", "neutral"),
    ("The color is a bit different from the photo, but I like it anyway.", "positive"),   # mixed
    ("Customer service never answered my emails.", "negative"),
    ("Not bad, not great. Probably fine for occasional use.", "neutral"),
    ("Comfortable, though a little pricey.", "neutral"),                                  # mixed
    ("Worst purchase this year. Returning it.", "negative"),
    ("Exceeded my expectations!", "positive"),
]
LABELS = ["positive", "negative", "neutral"]
print(len(REVIEWS), "reviews. Gold labels:", {l: sum(1 for _, g in REVIEWS if g == l) for l in LABELS})

## Part B — Eight prompts, one task

Every prompt asks the model to label sentiment. The differences are **harmless**: capital letters, the order of the options, where the instruction sits, extra blank lines, a role line. A robust system would not care.

In [ ]:
PROMPTS = {
    "V1 baseline":
        "Classify the sentiment of this review as positive, negative, or neutral.\n\nReview: {r}\nSentiment:",
    "V2 CAPS labels":
        "Classify the sentiment of this review as POSITIVE, NEGATIVE, or NEUTRAL.\n\nReview: {r}\nSentiment:",
    "V3 options reordered":
        "Classify the sentiment of this review as neutral, negative, or positive.\n\nReview: {r}\nSentiment:",
    "V4 reworded":
        "What is the overall feeling expressed in this review? Choose exactly one: positive, negative, neutral.\n\n{r}",
    "V5 quotes + label":
        'Review:\n"""{r}"""\nLabel (positive, negative, or neutral):',
    "V6 extra spacing":
        "Please read the review below carefully.\n\n\n{r}\n\n\nNow answer with one word: positive, negative, or neutral.",
    "V7 instruction last":
        "{r}\n\nIs the sentiment of the review above positive, negative, or neutral? Answer with one word.",
    "V8 role line":
        "You are a customer-experience analyst. Label the review's sentiment as positive, negative, or neutral. Reply with the label only.\n\nReview: {r}",
}

def parse_label(text):
    """Return the first label word found in the model's reply."""
    t = text.lower()
    found = [(t.find(l), l) for l in LABELS if l in t]
    return min(found)[1] if found else "unparsed"

def classify(template, review, **kw):
    reply = say(template.format(r=review), temperature=kw.get("temperature", 0.0), max_tokens=10)
    return parse_label(reply)

predictions = {}      # prompt name -> list of labels
for name, tmpl in PROMPTS.items():
    predictions[name] = [classify(tmpl, r) for r, _ in REVIEWS]
    print("done", name)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

gold = [g for _, g in REVIEWS]
acc = {name: sum(p == g for p, g in zip(preds, gold)) / len(gold) for name, preds in predictions.items()}
acc_s = pd.Series(acc)
print((acc_s * 100).round(0).astype(int).astype(str) + "%")
print()
print(f"Best prompt : {acc_s.idxmax()} ({acc_s.max():.0%})")
print(f"Worst prompt: {acc_s.idxmin()} ({acc_s.min():.0%})")
print(f"Spread      : {(acc_s.max() - acc_s.min()) * 100:.0f} percentage points, from changes that did not alter the meaning")

acc_s.plot(kind="bar", color="#CC0000", figsize=(9, 3.5))
plt.ylim(0, 1.05); plt.ylabel("Accuracy"); plt.xticks(rotation=30, ha="right")
plt.title("Same task, eight harmless rewordings"); plt.tight_layout(); plt.show()

💬 **Discuss:** If you had tested only V1 and reported its accuracy to your boss, how wrong could that number be? What does one test prompt tell you? (Compare this to testing a *single* day of sales before forecasting a year.)

## Part C — Which reviews are unstable?
Some reviews get the same label from every prompt. Others flip. Show the full grid.

In [ ]:
grid = pd.DataFrame(predictions, index=[f"{i+1}. {r[:38]}" for i, (r, _) in enumerate(REVIEWS)])
grid.insert(0, "gold", gold)

def n_distinct(row):
    return len(set(row[1:]))
grid["distinct labels"] = grid.apply(n_distinct, axis=1)
print(grid.to_string())
print()
unstable = grid[grid["distinct labels"] > 1]
print(f"{len(unstable)} of {len(grid)} reviews received different labels depending on the prompt wording.")

💬 **Discuss:** Look at the unstable rows. Are they the **mixed** reviews (7 and 10), or clear ones? What does that tell you about where a model is most fragile: easy cases or borderline ones? Where do the *costly* errors usually live in business?

## Part D — Wording sensitivity vs. run-to-run randomness
Two different sources of wobble:
- **Prompt sensitivity:** same task, different words (Part B).
- **Randomness:** same words, run again with a higher temperature.

Run one prompt five times at temperature 1.0 on the first six reviews, and count how often the label changes between runs.

In [ ]:
# ✏️ CHANGE ME — try a different prompt name from PROMPTS, or a different temperature (0.0, 0.7, 1.0, 1.5)
PROMPT_TO_REPEAT = "V1 baseline"
TEMP = 1.0
RUNS = 5

changed = 0
for r, _ in REVIEWS[:6]:
    labels = [classify(PROMPTS[PROMPT_TO_REPEAT], r, temperature=TEMP) for _ in range(RUNS)]
    flag = "  <- varies" if len(set(labels)) > 1 else ""
    changed += len(set(labels)) > 1
    print(f"{r[:45]:47s} {labels}{flag}")
print()
print(f"{changed} of 6 reviews changed label across {RUNS} runs at temperature {TEMP}.")

## Part E — Do examples make the prompt more robust?
Add three worked examples ("few-shot") to a few of the prompt styles and see whether the spread shrinks.

In [ ]:
EXAMPLES = (
    "Examples:\n"
    "Review: Stopped working after a week. Awful. -> negative\n"
    "Review: It does the job. -> neutral\n"
    "Review: Best purchase I have made all year! -> positive\n\n"
)

few_acc = {}
for name in ["V1 baseline", "V3 options reordered", "V6 extra spacing", "V7 instruction last"]:
    tmpl = EXAMPLES + PROMPTS[name]
    preds = [classify(tmpl, r) for r, _ in REVIEWS]
    few_acc[name + " + examples"] = sum(p == g for p, g in zip(preds, gold)) / len(gold)

compare = pd.DataFrame({"without examples": {k: acc[k] for k in ["V1 baseline", "V3 options reordered", "V6 extra spacing", "V7 instruction last"]},
                        "with examples": {k.replace(" + examples", ""): v for k, v in few_acc.items()}})
print((compare * 100).round(0).astype(int))
print()
print(f"Spread without examples: {(compare['without examples'].max() - compare['without examples'].min()) * 100:.0f} points")
print(f"Spread with examples   : {(compare['with examples'].max() - compare['with examples'].min()) * 100:.0f} points")

## Part F — ✏️ Your own variant
Write a ninth prompt that you think is **clearer** than the baseline, and measure it against the same 12 reviews. Is it really better, or just different?

In [ ]:
# ✏️ CHANGE ME — your own wording. Keep {r} where the review goes.
MY_PROMPT = "Read this customer review and tell me whether the customer felt positive, negative, or neutral. Reply with that one word.\n\n{r}"

mine = [classify(MY_PROMPT, r) for r, _ in REVIEWS]
my_acc = sum(p == g for p, g in zip(mine, gold)) / len(gold)
print(f"Your prompt: {my_acc:.0%} accuracy.   Baseline: {acc['V1 baseline']:.0%}.   Range of the eight: {acc_s.min():.0%} to {acc_s.max():.0%}.")
print("Changed labels vs baseline on reviews:", [i + 1 for i, (a, b) in enumerate(zip(mine, predictions['V1 baseline'])) if a != b])

## What this means for your Project 3 audit
When you audit another team's assistant, **never test one phrasing**. For each probe, ask it in at least three different ways and record all of them. A result that depends on exact wording is a finding, and a team that reports a single accuracy number from a single prompt has not told its manager the whole story.

---
## ✅ Notebook 22c complete — Reflection (submit with your notebook)
1. What was the spread in accuracy across your eight prompts? Why is a single prompt's accuracy a risky number to report?
2. Which reviews were unstable, and what do they have in common?
3. Did examples reduce the spread? What does that suggest about how to build a reliable classification feature?
4. Explain the difference between prompt sensitivity and randomness in one sentence each.

*Next session: Evaluating AI Systems. You will build a small test harness so you do not have to judge outputs by hand.*